# HeadEntropy demo

A correct and a wrong answer from each of the five datasets, given by **Qwen3-1.7B**, scored with HeadEntropy.

For every attention head, the trace of the softmax Jacobian `tr(J) = 1 − Σⱼ pⱼ²` is averaged over the
answer tokens (one number per head, `H̄`). The training-free correctness score is `−mean(H̄)`: the lower
the head entropy, the more likely the answer is correct.


In [ ]:
import torch
import pandas as pd
from transformers import AutoModelForCausalLM, AutoTokenizer

from run.utils.utils_generation import extract_answer, extract_math_answer, compute_token_idx_qwen
from run.utils.triviaqa_evaluation import exact_match_score, metric_max_over_ground_truths
from run.utils.math_evaluation import math_exact_match_score

MODEL = "Qwen/Qwen3-1.7B"
tok = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(
    MODEL, dtype=torch.bfloat16, attn_implementation="eager", device_map="cuda",  # eager: the only path that returns attention weights
).eval()

## Questions from each dataset

Ten questions from the paper's evaluation runs of Qwen3-1.7B, with the prompts the model was given
there: per dataset, the correct answer whose heads were the sharpest (lowest `mean H̄`) and the wrong
answer whose heads were the most diffuse (highest `mean H̄`).


In [ ]:
# dataset -> outcome in the run -> system prompt, user prompt, accepted answers
EXAMPLES = {
  "TriviaQA": {
    "correct": {
      "system": "You are a trivia expert. Please answer questions in exactly this format:\nAnswer: [1-3 words only]\nCertainty: [0-100]",
      "user": "Question: Who wrote the 1866 novel ‘Crime and Punishment’?",
      "gold": [
        "fyodor mikhailovich dostoevsky",
        "fyoder dostoyevski",
        "fyodor mikhaylovich dostoyevsky",
        "fiódor dostoiévski",
        "fjodor dostojewski",
        "fyodor dostoevskii",
        "dostoyevskii",
        "dosteovsky",
        "fyodor dostoievski",
        "f m dostoevsky",
        "feodor dostojevskij",
        "fodor dostoevsky",
        "dostoyevski",
        "fyodor dostoevsky old",
        "fyodor dostoyevski",
        "fedor m dostoevsky",
        "heavenly christmas tree",
        "fedor dostoevsky",
        "fyodor mikhailovich dostoyevsky",
        "fyodor dostoevski",
        "dostoyevskey",
        "fjodor m dostojewski",
        "fyodr dostoevsky",
        "fydor dostoyevsky",
        "dostoyevsky",
        "fëdor mikhailovich dostoevskii",
        "dostoyevsky fyodor",
        "fedor dostoyevski",
        "dostoevskij",
        "dostojevsky",
        "fedor dostoyevsky",
        "dostoievsky",
        "feodor mikhailovich dostoyevsky",
        "dostoevsky",
        "dostojevski",
        "fyodor dostoevsky",
        "doestoyevski",
        "fyodor mikhailovich dostoevski",
        "dostojevskij",
        "fyodor m dostoyevsky",
        "feodor dostoyevsky",
        "feodor dostoievsky",
        "fyodor m dostoevsky",
        "fiodor dostoevsky",
        "достоевский",
        "fedor m dostoyevsky",
        "feodor mikhailovich dostoievsky",
        "fiodor dostoïevski",
        "fedor mikhailovich dostoevskii",
        "feodor dostoevsky",
        "dostoievski",
        "feodor dostojevski",
        "fyodor dostoyevsky",
        "fyoder dostoyevsky",
        "fydor dostoevsky",
        "fedor dostoevski",
        "fyoder doestoyevski",
        "fiodor dostoievski",
        "dostoevski"
      ]
    },
    "wrong": {
      "system": "You are a trivia expert. Please answer questions in exactly this format:\nAnswer: [1-3 words only]\nCertainty: [0-100]",
      "user": "Question: In darts, what is the highest possible checkout with three darts, finishing with a double?",
      "gold": [
        "one hundred and seventy",
        "170"
      ]
    }
  },
  "HotpotQA": {
    "correct": {
      "system": "You are a helpful assistant.Answer the question using the information in the provided passages.Please answer questions in exactly this format:\nAnswer: [1-5 words only]\nCertainty: [0-100]",
      "user": "Question:\nTile Kolup pretended to be a member of what House?\nContext:\nFrederick II (26 December 1194 – 13 December 1250; Sicilian: \"Fidiricu II\" , German: \"Friedrich II\" ) was a Holy Roman Emperor and King of Sicily in the Middle Ages, a member of the House of Hohenstaufen.\n His political and cultural ambitions, based in Sicily and stretching through Italy to Germany, and even to Jerusalem, were enormous.\n However, his enemies, especially the popes, prevailed, and his dynasty collapsed soon after his death.\nTile Kolup (died July 7, 1285), also known as Dietrich Holzschuh, was an impostor who in 1284 began to pretend to be the Emperor Frederick II.\nEldon Quinn Johnson (August 16, 1930 – September 4, 2015) was an American politician who was a Republican member of the Oregon House of Representatives.\n He was appointed in September 1977 to fill the House District 51 seat vacated by Representative Brad Morris.\n He was unable to continue to serve in Oregon Legislature due to terms limits enacted by voters in the 1992 General Election, later ruled unconstitutional by the Oregon Supreme Court in 2002.\n His 11th and final term in office ended January 10, 1999.\n Johnson was born in West Point, Nebraska and attended school in Tucson, Arizona.\n He was an electrical and plumbing supply company owner, tile setter and farmer.\n He also served in the United States Air Force.\nThe 1982 Kenyan coup d'état attempt was a failed attempt to overthrow President Daniel arap Moi's government.\n At midnight on Sunday, 1 August 1982, a group of soldiers from the Kenya Air Force took over the radio station Voice of Kenya and announced that they had overthrown the government.\n The group tried to force a group of Air Force fighter pilots to bomb the State House at gunpoint.\n The pilots pretended to follow orders on the ground but once airborne they ignored them (confusing a member the coup group in one of the plans) and instead dropped the bombs over Mount Kenya's forests.",
      "gold": [
        "Hohenstaufen"
      ]
    },
    "wrong": {
      "system": "You are a helpful assistant.Answer the question using the information in the provided passages.Please answer questions in exactly this format:\nAnswer: [1-5 words only]\nCertainty: [0-100]",
      "user": "Question:\nWhat date in 2010 was a South Korean film starring  Kim Hyang-gi released?\nContext:\nWedding Dress is a South Korean drama film, released on January 14, 2010.\nKim Hyang-gi (born August 9, 2000) is a South Korean actress.\n Kim began her career as a child actress, and has starred in films and television series such as \"Wedding Dress\" (2010), \"The Queen's Classroom\" (2013), \"Thread of Lies\" (2014) and \"Snowy Road\" (2017).\nThread of Lies (; lit.\n Elegant Lies) is a 2014 South Korean film based on the 2009 bestselling novel \"Elegant Lies\" by Kim Ryeo-ryeong.\n Directed by Lee Han, it starred Kim Hee-ae (in her first film in 21 years), Go Ah-sung, Kim Hyang-gi and Kim Yoo-jung.\nEleventh Mom (; also known as My 11th Mother) is a 2007 South Korean film starring Kim Hye-soo, Kim Young-chan and Ryu Seung-ryong.\n It was released on November 29, 2007 and attracted 350,204 admissions.\nLe Grand Chef 2: Kimchi Battle (, also known as Le Grand Chef 2: Kimchi Wars) is a 2010 South Korean film starring Kim Jung-eun and Jin Goo.\n It was released on January 28, 2010.\nThe Railroad () is a 2006 South Korean film starring Kim Kang-woo and Son Tae-young.\n The second feature film of writer and director Park Heung-sik, it was also co-produced and co-edited by his wife, Park Gok-ji.\n \"The Railroad\" won the FIPRESCI award and Best Actor for Kim Kang-woo at the 25th Torino Film Festival.\n The name is taken from the Gyeongui Line.\nBetween Love and Hate (also known as The Unbearable Lightness of Dating) is a 2006 South Korean film starring Kim Seung-woo and Jang Jin-young, and is the directorial debut of screenwriter Kim Hae-gon.\n Jang's performance won her Best Actress at the 2006 Korean Film Awards.\n This would be Jang Jin-young's final film before her death almost 3 years later.\nDetective K: Secret of the Virtuous Widow () is a 2011 South Korean film based on the novel by Kim Tak-hwan, starring Kim Myung-min in the lead role.\n It was the 4th best selling Korean film of 2011.\nCherry Tomato () is a 2008 South Korean film starring Shin Goo and Kim Hyang-gi.\n The family drama, a directorial debut by Jung Young-bae, depicts the poverty-stricken life of an old man and his granddaughter that evokes a strong sense of sympathy and helplessness.\n It was screened at the Busan Children’s Film Festival in 2008.\nAlong With The Gods – Part 1 () is an upcoming South Korean fantasy drama film based on a webcomic of the same name.\n The film will be released in two parts, and stars Ha Jung-woo, Cha Tae-hyun, Ju Ji-hoon, Lee Jung-jae, Do Kyung-soo and Kim Hyang-gi.\n The first part of the film will be released on December 20, 2017.",
      "gold": [
        "January 14, 2010"
      ]
    }
  },
  "MedMCQA": {
    "correct": {
      "system": "You are a medical expert. Please answer questions in exactly this format:\nAnswer: [repeat correct option]\nCertainty: [0-100]",
      "user": "Question: \nThe cyst that moves by protruding the tongue is:\nOptions: \nThyroglossal cyst\n Median rhomboid cyst\n Ranula\n Tracheal cyst",
      "gold": [
        "Thyroglossal cyst"
      ]
    },
    "wrong": {
      "system": "You are a medical expert. Please answer questions in exactly this format:\nAnswer: [repeat correct option]\nCertainty: [0-100]",
      "user": "Question: \nThe following system of depiction of 32 teeth is in accordance with which system?Permanent TeethUpper RightUpper Left1234567891011121314151632313029282726252423222120191817Lower RightLower Left\nOptions: \nUniversal system\n Palmer's system\n Haderup system\n Diagrammatic depiction",
      "gold": [
        "Universal system"
      ]
    }
  },
  "MATH": {
    "correct": {
      "system": "You are a math expert. Please answer questions in exactly this format:\nAnswer: [final answer only, e.g. 42, 3/5, sqrt(2)]\nCertainty: [0-100]",
      "user": "Question: Convert $\\frac{60}{7}$ to a mixed number.",
      "gold": [
        "8\\frac47"
      ]
    },
    "wrong": {
      "system": "You are a math expert. Please answer questions in exactly this format:\nAnswer: [final answer only, e.g. 42, 3/5, sqrt(2)]\nCertainty: [0-100]",
      "user": "Question: A Penteria is a special (fictional) kind of bacteria such that, regardless of the original population in a collection, the population increases by $5$ every minute. Additionally, at the end of every hour, all but the initial ones die. If $137$ Penteria are alive after $506$ minutes, how many were there at the beginning?",
      "gold": [
        "7"
      ]
    }
  },
  "FEVER": {
    "correct": {
      "system": "You are a fact-checking expert. Given a claim and evidence, decide whether the evidence supports the claim, refutes it, or contains not enough information. Please answer questions in exactly this format:\nAnswer: [SUPPORTS | REFUTES | NOT ENOUGH INFO]\nCertainty: [0-100]",
      "user": "Evidence: A staging area , or landing zone , is an intermediate storage area used for data processing during the extract , transform and load ( ETL ) process . A staging area ( otherwise staging point , staging base or staging post ) is a location where organisms , people , vehicles , equipment or material are assembled before use . In military usage , a staging area is a place where troops or equipment in transit are assembled or processed . Unlike normal bases , the facilities of a staging area are temporary , mainly because for a certain time it will hold much more troops and materiel than would be reasonable in peacetime . Militaries use staging areas to deploy military units , aircraft and warships and materiel ahead of an attack or invasion .\nClaim: A staging area is only an unused piece of land.",
      "gold": [
        "REFUTES"
      ]
    },
    "wrong": {
      "system": "You are a fact-checking expert. Given a claim and evidence, decide whether the evidence supports the claim, refutes it, or contains not enough information. Please answer questions in exactly this format:\nAnswer: [SUPPORTS | REFUTES | NOT ENOUGH INFO]\nCertainty: [0-100]",
      "user": "Evidence: Broken is the debut extended play ( EP ) and major-label debut by American industrial rock band Nine Inch Nails , released on September 22 , 1992 by Nothing Records , TVT Records , Interscope Records , and Atlantic Records . `` Happiness in Slavery '' is a song by American industrial rock band Nine Inch Nails from their debut extended play ( EP ) , Broken ( 1992 ) .\nClaim: Happiness in Slavery is from the country Broken.",
      "gold": [
        "NOT ENOUGH INFO"
      ]
    }
  }
}


## Generate an answer and compute HeadEntropy

In [ ]:
@torch.no_grad()
def answer_and_head_entropy(system, user, max_new_tokens=768):
    messages = [{"role": "system", "content": system}, {"role": "user", "content": user}]
    prompt = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=True)
    ids = tok(prompt, return_tensors="pt").input_ids.to(model.device)
    n_prompt = ids.shape[1]

    gen = model.generate(ids, max_new_tokens=max_new_tokens, do_sample=False, temperature=None, top_p=None, top_k=None)
    text = tok.decode(gen[0, n_prompt:], skip_special_tokens=True)

    idx = dict(sorted(compute_token_idx_qwen(gen[0].tolist(), tok).items(), key=lambda kv: kv[1]))
    boundaries = list(idx.values())
    answer_end = idx["answer_end"]
    answer_start = boundaries[boundaries.index(answer_end) - 1]

    # 3. one forward pass over prompt + generation for the attention maps
    out = model(gen, output_attentions=True)
    A = torch.stack(out.attentions, dim=0)[:, 0].float()          # [layers, heads, T, T], each row sums to 1
    tr_J = 1 - (A ** 2).sum(dim=-1)                               # [layers, heads, T]
    H_bar = tr_J[:, :, answer_start:answer_end].mean(dim=-1)      # [layers, heads]
    return text, H_bar.cpu()


In [ ]:
def short(user, width=90):
    q = user.split("Claim:")[-1] if "Claim:" in user else user.removeprefix("Question:")
    q = q.strip().split("\n")[0]
    return q[:width] + ("..." if len(q) > width else "")

rows, heatmaps = [], {}
for name, by_outcome in EXAMPLES.items():
    heatmaps[name] = {}
    for outcome, ex in by_outcome.items():
        text, H_bar = answer_and_head_entropy(ex["system"], ex["user"], max_new_tokens=768)
        gold = ex["gold"]
        answer_part = text.split("</think>")[-1] if "</think>" in text else ""
        if "answer:" not in answer_part.lower():
            pred, correct = "(no answer within 768 tokens)", False
        elif name == "MATH":
            pred = extract_math_answer(text)
            correct = metric_max_over_ground_truths(math_exact_match_score, pred, gold)
        else:
            pred = extract_answer(text)
            correct = metric_max_over_ground_truths(exact_match_score, pred.lower(), [g.lower() for g in gold])
        heatmaps[name][outcome] = H_bar
        rows.append({
            "dataset": name,
            "in the run": outcome,
            "question": short(ex["user"]),
            "model answer": pred,
            "gold": gold[0],
            "correct": bool(correct),
            "HeadEntropy score (−mean H̄)": -H_bar.mean().item(),
        })

pd.set_option("display.max_colwidth", 100)
pd.DataFrame(rows)


Higher score = the model's heads were sharper while writing the answer = more likely correct.
With one pair per dataset this is an illustration, not an evaluation; the paper's numbers come from
thousands of questions per dataset (`./run_*.sh`, then `./eval.sh`).


In [ ]:
import matplotlib.pyplot as plt

outcomes = ("correct", "wrong")
fig, axes = plt.subplots(2, len(heatmaps), figsize=(3.2 * len(heatmaps), 7), sharex=True, sharey=True,
                         gridspec_kw={"hspace": 0.3})
all_H = [H for d in heatmaps.values() for H in d.values()]
vmin, vmax = min(H.min() for H in all_H), max(H.max() for H in all_H)
for col, (name, by_outcome) in enumerate(heatmaps.items()):
    for row, outcome in enumerate(outcomes):
        ax = axes[row, col]
        H_bar = by_outcome[outcome]
        im = ax.imshow(H_bar, aspect="auto", cmap="viridis", vmin=vmin, vmax=vmax)
        ax.set_title(f"{name}, {outcome} in run\nmean H̄ = {H_bar.mean():.3f}")
        if row == 1: ax.set_xlabel("head")
        if col == 0: ax.set_ylabel("layer")
fig.colorbar(im, ax=axes, label="H̄ (tr J over answer tokens)", shrink=0.8)
plt.show()

**Figure.** Per-head HeadEntropy `H̄` of Qwen3-1.7B (28 layers × 16 heads) for one correct (top) and one
wrong (bottom) answer per dataset, averaged over the answer tokens. Correct answers show sharp heads
(dark, low `tr J`), most visibly in the late layers; wrong answers, here mostly generations that ran out of
budget while thinking, leave the heads diffuse (bright) across the middle layers. The training-free
score is `−mean(H̄)`, so a darker map means a higher score.
